In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 7 - WEEK 12 BAYESIAN OPTIMISATION
# Run from inside week12/
# ============================================================
#
# Week 11 produced a NEW incumbent.
#
# Week 10 had a severe GP calibration miss, but Week 11 was
# much better calibrated (~ -0.57 sigma) and the conservative
# local search produced the best observed value.
#
# Week 12 strategy:
# - refit including Week 11
# - centre on the NEW incumbent
# - keep the search local
# - compare posterior mean with low-beta UCB
# - treat higher-beta / EI primarily as diagnostics
# - inspect coordinate profiles before choosing the final point
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 12 data
# ------------------------------------------------------------

X = np.load(
    "function7/initial_inputs.npy"
)

Y = np.load(
    "function7/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 11 calibration check
# ------------------------------------------------------------

week11_pred_mean = (
    3.095186
)

week11_pred_std = (
    0.048891
)

week11_actual = (
    3.067532247418812
)

week11_error = (
    week11_actual
    - week11_pred_mean
)

week11_z_error = (
    week11_error
    / week11_pred_std
)

print("\n================================")
print("WEEK 11 CALIBRATION CHECK")
print("================================")

print(
    "Predicted mean:",
    week11_pred_mean
)

print(
    "Predicted std :",
    week11_pred_std
)

print(
    "Actual        :",
    week11_actual
)

print(
    "\nPrediction error:",
    week11_error
)

print(
    "\nError / predicted std:",
    week11_z_error
)


# ------------------------------------------------------------
# 3. Recent observations
# ------------------------------------------------------------

week9_x = np.array([
    0.165230,
    0.227995,
    0.587283,
    0.251481,
    0.277646,
    0.674437
])

week9_y = (
    3.0575674596116973
)

week10_x = np.array([
    0.183574,
    0.174926,
    0.690591,
    0.267285,
    0.296515,
    0.680694
])

week10_y = (
    2.8408984660368275
)

week11_x = np.array([
    0.174931,
    0.257671,
    0.577020,
    0.262318,
    0.287196,
    0.669103
])

week11_y = (
    3.067532247418812
)

print("\n================================")
print("RECENT LOCAL POINTS")
print("================================")

print("\nWeek 9:")
print(
    week9_x,
    "->",
    week9_y
)

print("\nWeek 10:")
print(
    week10_x,
    "->",
    week10_y
)

print("\nWeek 11 / current best:")
print(
    week11_x,
    "->",
    week11_y
)


# ------------------------------------------------------------
# 4. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(6) * 0.3,
        length_scale_bounds=(
            0.01,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(
            1e-8,
            1e-1
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(
    X,
    Y
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0
    / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)
print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask]
    - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# Week 11 succeeded with a conservative local move.
# Keep Week 12 in roughly the same spirit.

empirical_cap = min(
    1.25 * nearest_distance,
    0.055
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print(
    "Nearest observed point:",
    nearest_distance
)

print(
    "\nEmpirical cap:",
    empirical_cap
)


# ------------------------------------------------------------
# 7. ARD-informed trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.16 * lengthscales,
    0.018,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x
    - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x
    + trust_half_width
)

print("\n================================")
print("WEEK 12 TRUST REGION")
print("================================")

print("Centre:")
print(
    best_x
)

print(
    "\nHalf-widths:"
)
print(
    trust_half_width
)

print(
    "\nLower:"
)
print(
    lower
)

print(
    "\nUpper:"
)
print(
    upper
)


# ------------------------------------------------------------
# 8. Dense local candidate pool
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

candidates = rng.uniform(
    lower,
    upper,
    size=(
        550000,
        6
    )
)

tree = cKDTree(
    X
)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "\nCandidates after duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 9. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 10. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "mean =",
    mu[ei_idx]
)

print(
    "std =",
    sigma[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)


# ------------------------------------------------------------
# 11. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "mean =",
    mu[mean_idx]
)

print(
    "std =",
    sigma[mean_idx]
)


# ------------------------------------------------------------
# 12. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"beta={beta}",
        "\n candidate =",
        candidates[idx],
        "\n mean =",
        round(
            mu[idx],
            6
        ),
        "\n std =",
        round(
            sigma[idx],
            6
        ),
        "\n UCB =",
        round(
            UCB[idx],
            6
        ),
        "\n"
    )


# ------------------------------------------------------------
# 13. Distance from new incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 14. Trust-region boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(
        len(x)
    ):

        if abs(
            x[j]
            - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j]
            - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(
        status
    )


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )


# ------------------------------------------------------------
# 15. Axis profile diagnostics
# ------------------------------------------------------------

print("\n================================")
print("AXIS PROFILE DIAGNOSTICS")
print("================================")

for j in range(6):

    values = np.linspace(
        lower[j],
        upper[j],
        35000
    )

    profile = np.tile(
        best_x,
        (
            len(values),
            1
        )
    )

    profile[:, j] = values

    profile_distance, _ = tree.query(
        profile,
        k=1
    )

    profile_valid = profile[
        profile_distance > 0.01
    ]

    if len(profile_valid) == 0:

        print(
            f"\nx{j+1}: no admissible points"
        )

        continue

    profile_mu, profile_sigma = gp.predict(
        profile_valid,
        return_std=True
    )

    idx = np.argmax(
        profile_mu
    )

    print(
        f"\nx{j+1} profile best:"
    )

    print(
        "candidate =",
        profile_valid[idx]
    )

    print(
        "mean =",
        profile_mu[idx]
    )

    print(
        "std =",
        profile_sigma[idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            profile_valid[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 16. Week 9 -> Week 11 direction diagnostic
# ------------------------------------------------------------
#
# Week 9 was the previous best and Week 11 improved it.
#
# Search along the empirical direction joining those two strong
# observations. This tests whether the GP wants to continue
# past Week 11 or move back toward Week 9.
# ------------------------------------------------------------

direction = (
    week11_x
    - week9_x
)

direction = (
    direction
    / np.linalg.norm(direction)
)


# Work out maximum feasible distances in each direction
pos_limits = []
neg_limits = []

for j in range(6):

    if direction[j] > 0:

        pos_limits.append(
            (upper[j] - best_x[j])
            / direction[j]
        )

        neg_limits.append(
            (best_x[j] - lower[j])
            / direction[j]
        )

    elif direction[j] < 0:

        pos_limits.append(
            (lower[j] - best_x[j])
            / direction[j]
        )

        neg_limits.append(
            (best_x[j] - upper[j])
            / direction[j]
        )


t_pos_max = min(
    pos_limits
)

t_neg_max = min(
    neg_limits
)


t_values = np.linspace(
    -t_neg_max,
    t_pos_max,
    50000
)

line_candidates = (
    best_x
    + t_values[:, None]
    * direction
)


line_distance, _ = tree.query(
    line_candidates,
    k=1
)

line_valid = line_candidates[
    line_distance > 0.01
]


if len(line_valid) > 0:

    line_mu, line_sigma = gp.predict(
        line_valid,
        return_std=True
    )

    line_idx = np.argmax(
        line_mu
    )

    print("\n================================")
    print("WEEK 9 -> WEEK 11 DIRECTION")
    print("================================")

    print(
        "Best line candidate =",
        line_valid[line_idx]
    )

    print(
        "mean =",
        line_mu[line_idx]
    )

    print(
        "std =",
        line_sigma[line_idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            line_valid[line_idx]
            - best_x
        )
    )

else:

    print(
        "\nNo admissible Week 9 -> Week 11 line points."
    )

DATA
X shape: (41, 6)
Y shape: (41,)

Current best:
[0.174931 0.257671 0.57702  0.262318 0.287196 0.669103] -> 3.06753224741881

Y range:
min = 0.0027014650245082332
max = 3.06753224741881
std = 1.0502854219643238

WEEK 11 CALIBRATION CHECK
Predicted mean: 3.095186
Predicted std : 0.048891
Actual        : 3.067532247418812

Prediction error: -0.027653752581187963

Error / predicted std: -0.5656205146384399

RECENT LOCAL POINTS

Week 9:
[0.16523  0.227995 0.587283 0.251481 0.277646 0.674437] -> 3.0575674596116973

Week 10:
[0.183574 0.174926 0.690591 0.267285 0.296515 0.680694] -> 2.8408984660368275

Week 11 / current best:
[0.174931 0.257671 0.57702  0.262318 0.287196 0.669103] -> 3.067532247418812


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
0.625**2 * Matern(length_scale=[0.498, 0.412, 0.814, 0.408, 0.261, 0.393], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[0.49819056 0.41232819 0.81359744 0.40846553 0.26051182 0.39338102]

Normalised inverse-lengthscale sensitivity:
[0.138523   0.16736874 0.08482186 0.16895147 0.26490488 0.17543005]

EMPIRICAL LOCAL SCALE
Nearest observed point: 0.03629322486360228

Empirical cap: 0.04536653107950285

WEEK 12 TRUST REGION
Centre:
[0.174931 0.257671 0.57702  0.262318 0.287196 0.669103]

Half-widths:
[0.04536653 0.04536653 0.04536653 0.04536653 0.04168189 0.04536653]

Lower:
[0.12956447 0.21230447 0.53165347 0.21695147 0.24551411 0.62373647]

Upper:
[0.22029753 0.30303753 0.62238653 0.30768453 0.32887789 0.71446953]

Candidates after duplicate filtering:
549990

PRIMARY EI
candidate = [0.17335392 0.22595236 0.53178672 0.26874296 0.29782519 0.68345609]
mean = 3.0442385058364803
std = 0.07341346004020999
EI = 0.019102906825741867

HIGHEST PREDICTED ME

In [2]:
# ============================================================
# FINAL FUNCTION 7 - WEEK 12 SELECTION
# ============================================================
#
# Week 11 produced the new incumbent and restored good local
# GP calibration after the Week 10 miss.
#
# For Week 12:
# - the highest posterior-mean candidate is interior
# - beta=0.05 and beta=0.10 select the same nearby region
# - the candidate remains close to the incumbent
# - its posterior uncertainty is low
# - axis profiles support the main movement, especially lower x3
#
# EI and higher-beta UCB move farther toward the x3 boundary
# and are more uncertainty-driven, so they are not selected.
# ============================================================

week12_candidate = np.array([
    0.165836,
    0.256907,
    0.556250,
    0.254875,
    0.282301,
    0.673252
])

print("================================")
print("FUNCTION 7 - WEEK 12 FINAL")
print("================================")

print("\nCandidate:")
print(week12_candidate)

# GP prediction
final_mu, final_sigma = gp.predict(
    week12_candidate.reshape(1, -1),
    return_std=True
)

print("\nPredicted mean:")
print(final_mu[0])

print("\nPredicted std:")
print(final_sigma[0])

print("\nPredicted improvement over observed best:")
print(final_mu[0] - best_y)

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week12_candidate - best_x
    )
)

# Check portal-rounded candidate against all previous observations
nearest_dist, nearest_idx = tree.query(
    week12_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

print("\nNearest observed point:")
print(X[nearest_idx[0]])

print("\nDuplicate rule satisfied:")
print(nearest_dist[0] > 0.01)

portal = "-".join(
    f"{x:.6f}"
    for x in week12_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 7 - WEEK 12 FINAL

Candidate:
[0.165836 0.256907 0.55625  0.254875 0.282301 0.673252]

Predicted mean:
3.074907274538678

Predicted std:
0.009317602069796507

Predicted improvement over observed best:
0.00737502711986826

Distance from current best:
0.024723856818870252

Nearest observed-point distance:
0.024723856818870252

Nearest observed point:
[0.174931 0.257671 0.57702  0.262318 0.287196 0.669103]

Duplicate rule satisfied:
True

Portal format:
0.165836-0.256907-0.556250-0.254875-0.282301-0.673252
